In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoConfig, BitsAndBytesConfig, AutoModelForCausalLM, TextStreamer, TrainingArguments, Trainer
from huggingface_hub import login
from sklearn.model_selection import train_test_split
from datasets import Dataset
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModelForCausalLM
)
from trl import SFTTrainer

c:\Users\omen\Desktop\projects\gemma4-darija-dz\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# login to Hugging Face using the token from the .env file
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Warning: HF_TOKEN not found in environment")
model_id = "google/gemma-4-E2B"
device_map = {"": 0}
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

GPU: NVIDIA GeForce RTX 3070 Laptop GPU
VRAM: 7.99951171875 GB


In [4]:
# Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["vision_tower", "audio_tower"],
)

In [5]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [6]:
import gc
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_id, extra_special_tokens={})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

streamer = TextStreamer(tokenizer, skip_prompt=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={'': 0},
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    attn_implementation="sdpa",  # Accelerated PyTorch Scaled Dot-Product Attention
)


c:\Users\omen\Desktop\projects\gemma4-darija-dz\.venv\Lib\site-packages\transformers\modeling_utils.py:5136: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)
Loading weights: 100%|██████████| 1951/1951 [00:30<00:00, 64.05it/s] 


In [7]:
# skipped (diagnostic only)

In [8]:
# skipped (diagnostic only)

In [9]:
# skipped (diagnostic only)

In [10]:
print("Model loaded successfully. Skipping full model print to save memory.")

Model loaded successfully. Skipping full model print to save memory.


In [11]:
# Bypass prepare_model_for_kbit_training to avoid OOM on Gemma4's
# embed_tokens_per_layer (262144 x 8960 bf16 = ~4.7GB → fp32 = ~9.4GB → OOM)

# Step 1: Disable KV cache (required for gradient checkpointing)
model.config.use_cache = False

# Step 2: Enable gradient checkpointing directly
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Step 3: Selectively upcast small non-4bit params to float32
# (skip large embeddings that would exceed 8GB VRAM)
SKIP_UPCAST = {"embed_tokens_per_layer", "embed_tokens", "lm_head"}
for name, param in model.named_parameters():
    if param.__class__.__name__ == "Params4bit":
        continue  # quantized weights — leave as-is
    if param.dtype not in (torch.float16, torch.bfloat16):
        continue  # already float32 or other dtype
    if any(skip in name for skip in SKIP_UPCAST):
        continue  # too large to safely upcast on 8GB VRAM
    param.data = param.data.to(torch.float32)

# Step 4: Freeze all base-model parameters
for param in model.parameters():
    param.requires_grad_(False)

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after prep: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=r"model\.language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


VRAM after prep: 6.31 GB allocated
trainable params: 24,158,208 || all params: 5,128,455,712 || trainable%: 0.4711


In [12]:
df = pd.read_csv("arabic_v1.csv")
if "Text" in df.columns and "text" not in df.columns:
    df = df.rename(columns={"Text": "text"})
dataset = Dataset.from_pandas(df[["text"]].dropna())
splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]


In [13]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./gemma4-darija-qlora",
    dataset_text_field="text",
    max_length=512,
    packing=True,                   # Packs short texts into 512-token sequences (eliminates pad waste)
    dataset_num_proc=4,             # Pre-tokenizes & packs in parallel on 4 CPU cores
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16 packed chunks (~300-400 short sentences per step)
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_steps=20,                   # Quick test: Run exactly 20 steps to observe loss reduction
    logging_steps=2,                # Log loss every 2 steps to see the loss curve clearly
    eval_strategy="steps",
    eval_steps=10,                  # Evaluate at step 10 and step 20
    save_steps=20,
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none"
)


In [14]:
# model is already a PeftModel from get_peft_model() in cell 10.
# TRL 1.10 raises if you pass both a PeftModel AND peft_config — so omit peft_config.
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)


[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported Flash Attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported Flash Attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss


In [ ]:
trainer.model.save_pretrained("gemma4-darija-qlora")
tokenizer.save_pretrained("gemma4-darija-qlora")
print("Training finished & LoRA adapters saved!")

In [ ]:
# Load base model + LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map=device_map,
    torch_dtype=compute_dtype
)
model_with_adapter = PeftModelForCausalLM.from_pretrained(base_model, "./gemma4-darija-qlora")
prompt = "واش راك اليوم؟" # "How are you today?" in Darija
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model_with_adapter.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
